In [1]:
import torch
import torch.nn as nn
import numpy as np

file_path = r"C:\Users\alias\Downloads\input.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}

encoded = np.array([char2idx[c] for c in text])

In [3]:
seq_length = 100

X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i+seq_length])
    y.append(encoded[i+seq_length])

X = np.array(X)
y = np.array(y)

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(X.shape, y.shape)

torch.Size([1115294, 100]) torch.Size([1115294])


In [4]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, 256)
        
        self.gru = nn.GRU(
            input_size=256,
            hidden_size=512,
            num_layers=2,
            batch_first=True
        )
        
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        
        out, _ = self.gru(x)
        
        out = out[:, -1, :]
        
        out = self.dropout(out)
        out = self.fc(out)
        
        return out

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GRUModel(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
from tqdm import tqdm
import time
batch_size = 64
epochs = 5

for epoch in range(epochs):
    total_loss = 0
    
    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size].to(device)
        yb = y[i:i+batch_size].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criterion(outputs, yb)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
for i in tqdm(range(100)):
    time.sleep(0.05)

KeyboardInterrupt: 

In [ ]:
def generate(model, start_text, length=300):
    model.eval()
    
    input_seq = torch.tensor(
        [char2idx[c] for c in start_text],
        dtype=torch.long
    ).unsqueeze(0).to(device)
    
    generated = start_text
    
    for _ in range(length):
        output = model(input_seq)
        probs = torch.softmax(output, dim=-1)
        
        char_id = torch.argmax(probs).item()
        
        generated += idx2char[char_id]
        
        input_seq = torch.cat([
            input_seq[:, 1:], 
            torch.tensor([[char_id]]).to(device)
        ], dim=1)
    
    return generated


print(generate(model, "First Citizen:", 300))

In [ ]:
import math

perplexity = math.exp(loss.item())
print("Perplexity:", perplexity)